In [ ]:
# Ô khởi động SSH cho Claude: chạy MỘT lần mỗi phiên Colab, rồi dán dòng "HOST: ..." vào chat.
# Chỉ đăng nhập được bằng khóa SSH của Claude trên máy Nhật Anh (không có mật khẩu).
import os, re, time, shlex, pathlib, subprocess

PUB = "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIMu0SMQ2TYz1dLWLwfmwXBt1dS+A+cNMOJ6iu3HBjoO6 colab-8b"

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

# 1) GPU
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "KHONG CO GPU")

# 2) Drive (để lưu checkpoint/kết quả phòng khi mất phiên) - bấm cho phép khi được hỏi
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/nlp_8b", exist_ok=True)

# 3) SSH chỉ bằng khóa
sh("apt-get -qq update > /dev/null && apt-get -qq install -y openssh-server tmux rsync > /dev/null")
ssh_dir = pathlib.Path("/root/.ssh"); ssh_dir.mkdir(mode=0o700, exist_ok=True)
(ssh_dir / "authorized_keys").write_text(PUB + "\n"); os.chmod(ssh_dir / "authorized_keys", 0o600)
with open("/etc/ssh/sshd_config", "a") as f:
    f.write("\nPasswordAuthentication no\nKbdInteractiveAuthentication no\nPermitRootLogin prohibit-password\nPubkeyAuthentication yes\n")
sh("mkdir -p /run/sshd && (pgrep -x sshd > /dev/null || /usr/sbin/sshd)")

# 4) Cho phiên SSH dùng đúng môi trường Python/CUDA của Colab
skip = {"HOME", "PWD", "SHLVL", "_", "TERM", "OLDPWD"}
with open("/root/.colab_env", "w") as f:
    for k, v in os.environ.items():
        if k not in skip and re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", k):
            f.write(f"export {k}={shlex.quote(v)}\n")
sh("grep -q colab_env /root/.bashrc || echo 'source /root/.colab_env; cd /content' >> /root/.bashrc")

# 5) Đường hầm cloudflared (không cần tài khoản)
sh("test -x /usr/local/bin/cloudflared || (wget -q -O /usr/local/bin/cloudflared "
   "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
   "&& chmod +x /usr/local/bin/cloudflared)")
log = "/content/cloudflared.log"
subprocess.Popen(f"nohup cloudflared tunnel --no-autoupdate --url ssh://localhost:22 > {log} 2>&1 &", shell=True)
host = None
for _ in range(90):
    time.sleep(2)
    m = re.search(r"https://([a-z0-9-]+\.trycloudflare\.com)", open(log).read())
    if m:
        host = m.group(1); break
print("HOST:", host or "CHUA CO - chay lai o nay")

# 6) Giữ ô chạy để phiên không bị coi là rảnh (Pro+: bật chạy nền rồi có thể đóng tab)
while True:
    time.sleep(300)
    print(time.strftime("%H:%M"), "con song", flush=True)
